In [3]:
# ---------------- code impératif avec sqlite3 -------------
"""
SQLITE3: BDD relationnelle avec un dialecte SQL simple
BDD en un seul fichier ou chargé en RAM
pas de host / port / user / password / droits
le module sqlite3 python est dans la lib standard

sqlite3 utilise la DbAPI python pour tous les clients de BDD en python
- méthode connect() => connexion (création ou connexion sur la bdd)
- méthode cursor()  => prompt sur la connexion
- méthode execute() => exécuter une requête SQL sur le cursor
- méthode fetch()   => faire sortir les enregistrements attendus
- méthode close()   => fermer la cnx
---------------- gérer les transactions -----------------
- méthode begin()   => commence une transaction
- methode commit()  => confitmer une tr
- méthode rollback()=> annuler une tr
"""

import sqlite3

conn = sqlite3.connect("dns.db")
cur  = conn.cursor()
req = "SELECT SQLITE_VERSION()"
cur.execute(req)
row  = cur.fetchone()
print(row)
conn.close()

('3.49.1',)


In [9]:
# ----------------- une connexion s'ouvre et se ferme ------------------
# exécuter un script SQL pour créer les tables de la bdd

with sqlite3.connect("dns.db") as conn:
    conn.row_factory = sqlite3.Row
    cur = conn.cursor()
    with open("./domain_names_sqlite3.sql", "r", encoding="utf-8") as f:
        sql_script = f.read()
        cur.executescript(sql_script)

    # test
    cur.execute("SELECT * FROM pays")
    rows = list(map(dict, cur.fetchall()))
    print(rows)


[{'iso2': 'AE', 'name': 'Emirats Arabes Unis'}, {'iso2': 'AT', 'name': 'Autriche'}, {'iso2': 'AU', 'name': 'Australie'}, {'iso2': 'BE', 'name': 'Belgique'}, {'iso2': 'BS', 'name': 'Bahamas'}, {'iso2': 'CA', 'name': 'Canada'}, {'iso2': 'CH', 'name': 'Chine'}, {'iso2': 'CY', 'name': 'Chypre'}, {'iso2': 'CZ', 'name': 'République tchèque'}, {'iso2': 'DE', 'name': 'Allemagne'}, {'iso2': 'DK', 'name': 'Danemark'}, {'iso2': 'ES', 'name': 'Espagne'}, {'iso2': 'FR', 'name': 'France'}, {'iso2': 'GB', 'name': 'Royaume Uni'}, {'iso2': 'GP', 'name': 'Guadeloupe'}, {'iso2': 'HK', 'name': 'Hong Kong'}, {'iso2': 'IE', 'name': 'Irlande'}, {'iso2': 'IL', 'name': 'Israël'}, {'iso2': 'IT', 'name': 'Italie'}, {'iso2': 'JP', 'name': 'Japon'}, {'iso2': 'KR', 'name': 'République de Corée'}, {'iso2': 'KY', 'name': 'Iles Caïmans'}, {'iso2': 'LI', 'name': 'Liechtenstein'}, {'iso2': 'LU', 'name': 'Luxembourg'}, {'iso2': 'MA', 'name': 'Maroc'}, {'iso2': 'MC', 'name': 'Monaco'}, {'iso2': 'MQ', 'name': 'Martinique'}

In [10]:
# ----------------------- insérer 100k lignes à partir d'un csv -----------------

# exemple de requête INSERT
#                                                               fixe                delim1                 delim2                    fixe
# INSERT INTO <nom_table> (<field_name1>, <field_name2>) VALUES ('  <field_value_1>  ','   <field_value_2> '), (' ...', '...'), ...  ')
# alternative: requêtes préparées
# "INSERT INTO <nom_table> (<field_name1>, <field_name2>) VALUES (?,?)"
# utiliser la méthode executemany()
import sqlite3
from pathlib import Path
import csv

p_slice = Path("./data/dns_100000.csv")
with open(p_slice, "r", encoding="utf-8") as f:
    reader = csv.reader(f, delimiter=";")
    # data = []
    next(reader)
    # for row in reader:
    #     data.append(row[:2])
    data = list(map(lambda r: r[:2], reader))

# print(data[:10])

## requête préparée
# prepared_req = "INSERT INTO domain_name (iso2, name) VALUES (?,?)"
# with sqlite3.connect("dns.db") as conn:
#     conn.row_factory = sqlite3.Row
#     cur = conn.cursor()
#     cur.executemany(prepared_req, data)
#     print(cur.rowcount)

## avec une seule requête
s_data = list(map(lambda r: "','".join(r), data))
s_data = "'),('".join(s_data)

req = f"INSERT INTO domain_name (iso2, name) VALUES ('{s_data}')"
with sqlite3.connect("dns.db") as conn:
    conn.row_factory = sqlite3.Row
    cur = conn.cursor()
    cur.execute(req)
    print(cur.rowcount)
## test
# après l'insertion on regarde l'attrit cur.rowcount (nb éléments modifiés)

100000


In [ ]:
# ----------------- client sqlite3 en poo et gestionnaire de contexte

In [ ]:
# -------------------- initiation à sqlAlchemy (véritable ORM) ---------------

# pip install SQLAlchemy

# connexion et session pour pooling
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker, Session

# définition des modèles
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship
# champs SQL
from sqlalchemy import String, Enum, Integer, Float, DateTime, Text, ForeignKey, func

from pathlib import Path

# -------------------------------- GLOBALS ------------------------------------

DB_PATH = Path(__file__).parent / "dns.db"
# chaîne de cnx :URI
DB_URI = f"sqlite:///{DB_PATH}"


engine = create_engine(
  DB_URI,
  # par défaut: on peut réutiliser le même thread pour une nouvelle cnx => pooling possible
  connect_args={"check_same_thread": False},
  # voir les requêtes SQL pour debug
  echo=True
)

# factory de sessions
SessionLocal = sessionmaker(autoflush=False, autocommit=False, bind=engine)

# --------------------------- MODELES --------------------------------------------

class Base(DeclarativeBase):
  """classe de base pour tous les modèles (tables)"""
  pass

class DomainName(Base):
  __tablename__ = "domain_name"

  id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
  iso2: Mapped[str] = mapped_column(String(2), nullable=False)
  name: Mapped[str] = mapped_column(String(100), unique=True, nullable=False, index=True)

class Pays(Base):
  __tablename__ = "pays"
  iso2: Mapped[str] = mapped_column(String(2), primary_key=True)
  name: Mapped[str] = mapped_column(String(100), unique=True, nullable=False)

# ------------------------------- init_db --------------------------------------------------


def init_db():
  import re
  import csv

  Base.metadata.create_all(bind=engine)

  with SessionLocal() as session:
    # Import pays depuis le script SQL
    sql_path = Path(__file__).parent / "domain_names_sqlite3.sql"
    with open(sql_path, "r", encoding="utf-8") as f:
      sql_content = f.read()

    match = re.search(r"INSERT INTO `pays` VALUES (.+);", sql_content)
    if match:
      pays_values = re.findall(r"\('([^']+)','([^']+)'\)", match.group(1))
      for iso2, name in pays_values:
        # Pays(iso2=iso2, name=name)  => objet enregistrement de table
        # cet objet a une réalité en python et aussi dans la base
        # il faut synchroniser régulièrement les objet py et sql flush() commit()
        session.merge(Pays(iso2=iso2, name=name))

    # Import domain_name depuis dns_10.csv
    csv_path = Path(__file__).parent / "dns_10.csv"
    with open(csv_path, mode="r", encoding="utf-8") as f:
      reader = csv.DictReader(f, delimiter=";")
      for row in reader:
        session.add(DomainName(name=row["Nom de domaine"], iso2=row["Pays BE"]))

    session.commit()

init_db()

In [ ]:
# ------------------------- usage ----------------------------------------
from sqlalchemy import select, func

with SessionLocal() as session:
  # SELECT * FROM domain_name
  dns_count = session.execute(select(func.count()).select_from(DomainName)).scalar()
  print(f"Nombre de domaines: {dns_count}")